In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import tensorflow_decision_forests as tfdf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")


In [ ]:
train = pd.read_csv("/kaggle/input/spaceship-titanic/train.csv")
test  = pd.read_csv("/kaggle/input/spaceship-titanic/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

test_ids = test["PassengerId"]


In [ ]:
def engineer(df):

    df = df.copy()

    # Cabin split
    df[["Deck","CabinNum","Side"]] = df["Cabin"].str.split("/", expand=True)
    df.drop("Cabin", axis=1, inplace=True)

    # Group features
    df["Group"] = df["PassengerId"].str.split("_").str[0]
    df["GroupSize"] = df.groupby("Group")["Group"].transform("count")

    # Spending features
    spend = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]
    df["TotalSpend"] = df[spend].sum(axis=1)
    df["HasSpent"] = (df["TotalSpend"] > 0).astype(int)

    # Age bins
    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[0,12,18,30,50,100],
        labels=["Child","Teen","Young","Adult","Senior"]
    )

    # Boolean → int
    for col in ["CryoSleep","VIP"]:
        df[col] = df[col].fillna(False).astype(int)

    # Numeric fill
    num_cols = df.select_dtypes(include=np.number).columns
    df[num_cols] = df[num_cols].fillna(0)

    # Convert categorical → string (TFDF compatible)
    cat_cols = df.select_dtypes(include=["object","category"]).columns
    df[cat_cols] = df[cat_cols].astype(str).fillna("Unknown")

    return df


train = engineer(train)
test  = engineer(test)


In [ ]:
train["Transported"] = train["Transported"].astype(int)

DROP = ["PassengerId","Name"]
train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)


In [ ]:
X = train.drop("Transported", axis=1)
y = train["Transported"]

SEEDS = [42,7,2024]
test_preds = []

for seed in SEEDS:

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_preds = []

    for fold,(tr,val) in enumerate(kf.split(X,y)):

        train_df = pd.concat([X.iloc[tr],y.iloc[tr]],axis=1)
        val_df   = pd.concat([X.iloc[val],y.iloc[val]],axis=1)

        train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_df,label="Transported")
        val_ds   = tfdf.keras.pd_dataframe_to_tf_dataset(val_df,label="Transported")
        test_ds  = tfdf.keras.pd_dataframe_to_tf_dataset(test)

        model = tfdf.keras.GradientBoostedTreesModel(
            num_trees=800,
            max_depth=7,
            shrinkage=0.03,
            subsample=0.85,
            sampling_method="RANDOM",
            random_seed=seed
        )

        model.fit(train_ds)

        preds = model.predict(val_ds)
        preds = (preds>0.5).astype(int).squeeze()
        acc = accuracy_score(y.iloc[val],preds)
        print(f"Seed {seed} Fold {fold+1} Accuracy:",acc)

        fold_preds.append(model.predict(test_ds))

    test_preds.append(np.mean(fold_preds,axis=0))


In [ ]:
final_preds = np.mean(test_preds,axis=0)
final_preds = (final_preds>0.5).astype(bool).squeeze()
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": final_preds
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.head()
